<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session06_embeddings_and_vector_search_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۶: امبدینگ و جستجوی معنایی

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session06_embeddings_and_vector_search.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install sentence-transformers chromadb
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

import time

import numpy as np
import pandas as pd
from llm_utils import load_store_docs
from sentence_transformers import SentenceTransformer

docs = load_store_docs()
passages = []
for source, text in docs.items():
    title = text.split("\n", 1)[0].lstrip("# ").strip()
    for paragraph in text.split("\n\n"):
        paragraph = paragraph.strip()
        if len(paragraph) > 40 and not paragraph.startswith("#"):
            passages.append({"source": source, "title": title, "text": paragraph})
passages = pd.DataFrame(passages)

e5 = SentenceTransformer("intfloat/multilingual-e5-base")


def encode(model, texts, prefix=""):
    return model.encode([prefix + t for t in texts], normalize_embeddings=True, show_progress_bar=False)


def keyword_rank(query):
    words = set(query.split())
    return np.argsort(-np.array([len(words & set(t.split())) for t in passages.text]))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

پیشوندهای `query:` و `passage:` را حذف کنید و نتیجه پنج پرسش را با حالت درست مقایسه کنید. شباهت‌ها و ترتیب نتایج چه تغییری کرد؟

</div>

In [ ]:
queries = ["پولم کی برمی‌گرده؟", "اگه خونه نباشم چی میشه؟", "قسطی می‌شه خرید کرد؟",
           "رمز یک‌بار مصرف چقدر اعتبار داره؟", "نصب کولر گازی"]
texts = passages.text.tolist()
with_prefix = encode(e5, texts, "passage: ")
without_prefix = encode(e5, texts)

for q in queries:
    s1 = with_prefix @ encode(e5, [q], "query: ")[0]
    s2 = without_prefix @ encode(e5, [q])[0]
    print(f"\n{q}")
    print(f"  with prefix   : {s1.max():.3f}  {passages.title[s1.argmax()]}")
    print(f"  without prefix: {s2.max():.3f}  {passages.title[s2.argmax()]}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

مدل‌های E5 در آموزش همیشه پیشوند دیده‌اند. پیشوند `query:` به مدل می‌گوید این متن یک «پرسش کوتاه» است که باید به «متنی که پاسخ را دارد» نزدیک شود، و `passage:` نقش متن سند را مشخص می‌کند. بدون پیشوند، ورودی با توزیع داده‌های آموزشی فرق دارد. معمولاً همهٔ شباهت‌ها کمی جابه‌جا می‌شوند و در پرسش‌های دشوارتر ترتیب نتایج هم عوض می‌شود.

با پنج پرسش ممکن است تفاوت در نتیجهٔ اول دیده نشود. برای سنجش واقعی، از مجموعهٔ ارزیابی با Hit Rate استفاده کنید (تمرین ۲ و جلسهٔ ۹). قاعدهٔ کلی: **همیشه قرارداد مدل را از Model Card آن بخوانید.**

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

ده پرسش محاوره‌ای درباره سیاست‌های نوین‌کالا بنویسید و برای هر کدام بررسی کنید سند درست در سه نتیجه اول جستجوی معنایی و کلیدواژه‌ای هست یا نه. کدام بهتر بود؟

</div>

In [ ]:
QUESTIONS = [
    ("پولم کی برمی‌گرده؟", "02_returns.md"),
    ("اگه خونه نباشم چی میشه؟", "01_shipping.md"),
    ("میشه قسطی خرید؟", "04_payment.md"),
    ("گوشیم افتاد شکست، گارانتی داره؟", "03_warranty.md"),
    ("امتیازامو چیکار کنم؟", "05_club.md"),
    ("آدرسمو اشتباه زدم", "06_orders.md"),
    ("کد پیامکی چقدر اعتبار داره؟", "07_account.md"),
    ("کی میاد تلویزیونو نصب کنه؟", "08_installation.md"),
    ("جمعه زنگ بزنم کسی جواب میده؟", "10_support.md"),
    ("دو تا کد تخفیف با هم میشه زد؟", "11_gift_cards.md"),
]


def hit_at_3(rank_fn):
    hits = [source in set(passages.source.iloc[rank_fn(q)[:3]]) for q, source in QUESTIONS]
    return np.mean(hits), hits

passage_vectors = encode(e5, passages.text.tolist(), "passage: ")


def semantic_rank(query):
    return np.argsort(-(passage_vectors @ encode(e5, [query], "query: ")[0]))


sem, sem_hits = hit_at_3(semantic_rank)
key, key_hits = hit_at_3(keyword_rank)
print(f"hit@3  semantic: {sem:.0%}   keyword: {key:.0%}")
pd.DataFrame({"question": [q for q, _ in QUESTIONS], "semantic": sem_hits, "keyword": key_hits})

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پرسش‌ها عمداً محاوره‌ای‌اند: «پولم کی برمی‌گرده» به جای «بازپرداخت»، «کد پیامکی» به جای «رمز یک‌بار مصرف». جستجوی کلیدواژه‌ای ساده در این نوع پرسش‌ها معمولاً ضعیف است، چون کلمات پرسش در سند نیستند. جستجوی معنایی این فاصله را پر می‌کند.

ولی جستجوی کلیدواژه‌ای ما خیلی ابتدایی است: نه نرمال‌سازی دارد، نه وزن‌دهی IDF و نه توکن‌سازی درست. در جلسهٔ ۹ با BM25 مقایسه‌ای منصفانه‌تر انجام می‌دهیم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

مدل `BAAI/bge-m3` را جایگزین کنید (این مدل پیشوند لازم ندارد) و تمرین ۲ را تکرار کنید. کیفیت و سرعت را مقایسه کنید.

</div>

In [ ]:
QUESTIONS = [
    ("پولم کی برمی‌گرده؟", "02_returns.md"),
    ("اگه خونه نباشم چی میشه؟", "01_shipping.md"),
    ("میشه قسطی خرید؟", "04_payment.md"),
    ("گوشیم افتاد شکست، گارانتی داره؟", "03_warranty.md"),
    ("امتیازامو چیکار کنم؟", "05_club.md"),
    ("آدرسمو اشتباه زدم", "06_orders.md"),
    ("کد پیامکی چقدر اعتبار داره؟", "07_account.md"),
    ("کی میاد تلویزیونو نصب کنه؟", "08_installation.md"),
    ("جمعه زنگ بزنم کسی جواب میده؟", "10_support.md"),
    ("دو تا کد تخفیف با هم میشه زد؟", "11_gift_cards.md"),
]


def hit_at_3(rank_fn):
    hits = [source in set(passages.source.iloc[rank_fn(q)[:3]]) for q, source in QUESTIONS]
    return np.mean(hits), hits

bge = SentenceTransformer("BAAI/bge-m3")

rows = []
for name, model, q_prefix, p_prefix in [("multilingual-e5-base", e5, "query: ", "passage: "),
                                        ("bge-m3", bge, "", "")]:
    start = time.time()
    vectors = encode(model, passages.text.tolist(), p_prefix)
    index_seconds = time.time() - start
    rate, _ = hit_at_3(lambda q: np.argsort(-(vectors @ encode(model, [q], q_prefix)[0])))
    rows.append({"model": name, "dimension": vectors.shape[1], "hit@3": rate,
                 "index_seconds": round(index_seconds, 2)})
pd.DataFrame(rows)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`bge-m3` حدود ۵۶۸ میلیون پارامتر دارد، تقریباً دو برابر `e5-base`، و بعد بردارش ۱۰۲۴ است. پس ساخت شاخص کندتر است و حافظهٔ بیشتری می‌گیرد. مزیت‌های آن: کیفیت معمولاً بهتر در زبان‌های غیرانگلیسی و پذیرش ورودی تا ۸۱۹۲ توکن، که برای تکه‌های بلند مفید است.

با ده پرسش و ۳۵ پاراگراف، هر دو مدل احتمالاً به نتیجهٔ نزدیک به کامل می‌رسند و تفاوت دیده نمی‌شود. این خودش درس مهمی است: **روی مسئلهٔ ساده، مدل بزرگ‌تر لزوماً ارزش هزینه‌اش را ندارد.** تفاوت مدل‌ها روی مجموعه‌های بزرگ‌تر و پرسش‌های دشوارتر ظاهر می‌شود.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

در Chroma یک فیلد متادیتای جدید به نام `topic` با مقدارهای «مالی» و «لجستیک» بسازید و جستجویی بنویسید که فقط در اسناد مالی جستجو کند.

</div>

In [ ]:
import chromadb

TOPIC = {"04_payment.md": "مالی", "05_club.md": "مالی", "11_gift_cards.md": "مالی",
         "12_corporate.md": "مالی", "02_returns.md": "مالی",
         "01_shipping.md": "لجستیک", "06_orders.md": "لجستیک", "08_installation.md": "لجستیک"}

subset = passages[passages.source.isin(TOPIC)].reset_index(drop=True)
chroma = chromadb.Client()
collection = chroma.get_or_create_collection("novinkala_topics", metadata={"hnsw:space": "cosine"})
collection.upsert(
    ids=[f"p{i}" for i in range(len(subset))],
    documents=subset.text.tolist(),
    embeddings=encode(e5, subset.text.tolist(), "passage: ").tolist(),
    metadatas=[{"source": s, "topic": TOPIC[s]} for s in subset.source],
)

query = "چند روز طول می‌کشد؟"
for topic in [None, "مالی", "لجستیک"]:
    result = collection.query(query_embeddings=encode(e5, [query], "query: ").tolist(), n_results=2,
                              where={"topic": topic} if topic else None)
    print(f"\n--- topic={topic}")
    for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
        print(f"[{meta['topic']} | {meta['source']}] {doc[:80]}...")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پرسش «چند روز طول می‌کشد؟» بدون زمینه مبهم است: ممکن است دربارهٔ ارسال، بازپرداخت یا تعمیر باشد. فیلتر متادیتا با محدود کردن جستجو به یک موضوع، این ابهام را حل می‌کند.

در برنامهٔ واقعی، فیلتر را معمولاً از زمینهٔ گفتگو یا رابط کاربری می‌گیریم، مثلاً کاربر در صفحهٔ «پیگیری سفارش» است، یا با یک طبقه‌بند کوچک موضوع پرسش را تشخیص می‌دهیم. برچسب موضوع را هم بهتر است هنگام ساخت شاخص، از ساختار اسناد بگیریم، نه با دست.

</div>